## Step 9 — select one candidate subdivision per source block
**# of cells in notebook:** 1

**Purpose:** Select a single candidate block subdivision for each source block from the populated k = 2–5 alternatives created in Step 8. The selection criterion depends on whether the original block entered the workflow because of the LargePop criterion or a heterogeneity criterion.

**Input:**

- `new_blocks_populated.gpkg` from Step 8 for each source block
- `heterogeneous_largePop_blocks` from Step 1, including:
  - `LargePop`
  - `HH_CC`
  - `HH_Grtr10ha`
  - `CC_Grtr10ha`

**Output:**

- `heterogeneous_largePop_selection` directory containing one folder for each selected source block
- within each block folder:
  - `new_blocks_populated.gpkg` containing the selected candidate layer
- `selection_log.txt`
- `selection_summary.csv`

**Main logic:**

**Cell 1 — Select the preferred k solution**

1. Reads the original screening flags for each source block and identifies the available k = 2–5 candidate layers.
2. Evaluates the population and area of every feature in each candidate layer.
3. For blocks with `LargePop = 1`, selects the first available k, checking k = 2 through k = 5, for which every new block has population below 1,000.
4. If no LargePop candidate satisfies the population threshold, uses k = 5 if all k = 5 features are below 100,000 m²; otherwise still uses k = 5 and records a warning.
5. For heterogeneous blocks that are not LargePop, selects the first k for which every new block is below 100,000 m².
6. If no heterogeneous candidate satisfies the area threshold, uses k = 5 and records a warning.
7. Copies the selected layer into `heterogeneous_largePop_selection` and records the decision and candidate statistics in the selection summary.


In [ ]:
# -*- coding: utf-8 -*-
r"""
Select one populated new-block layer per source block, without ArcPy.

Input root:
    E:\_johannesburg\_analysis\heterogeneous_largePop_blocks

Each block folder is expected to contain:
    new_blocks_populated.gpkg

with layers such as:
    blk_1_442_2
    blk_1_442_3
    blk_1_442_4
    blk_1_442_5

Selection rules
---------------
For each block folder, consult:

    E:\_johannesburg\_analysis\blocks\blocks.gdb\heterogeneous_largePop_blocks

using:
    folder _442 -> block_id blk_442

If LargePop == 1:
    - Select the first layer, checking k=2, then k=3, then k=4, then k=5,
      where all feature populations are below 1000.
    - If none pass the population rule, select k=5 if all k=5 feature areas
      are below 100,000 m2.
    - If k=5 also fails the area rule, still select k=5 and log a warning.

If LargePop != 1 and any of HH_CC, HH_Grtr10ha, or CC_Grtr10ha == 1:
    - Select the first layer, checking k=2, then k=3, then k=4, then k=5,
      where all feature areas are below 100,000 m2.
    - If none pass the area rule, still select k=5 and log a warning.

Outputs
-------
Output root:
    E:\_johannesburg\_analysis\heterogeneous_largePop_selection

For each selected block:
    <output_root>\_<block_id>\new_blocks_populated.gpkg

The output GeoPackage contains the single selected layer for that block.

Also writes:
    selection_summary.csv
    selection_log.txt

Required packages:
    geopandas, pandas, pyogrio
"""

import traceback
from pathlib import Path
from datetime import datetime

import pandas as pd
import geopandas as gpd
import pyogrio


# ------------------------------------------------------------
# User settings
# ------------------------------------------------------------

INPUT_ROOT = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks"
)

OUTPUT_ROOT = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_selection"
)

SOURCE_BLOCKS_GDB = Path(
    r"E:\_johannesburg\_analysis\blocks\blocks.gdb"
)

SOURCE_BLOCKS_LAYER = "heterogeneous_largePop_blocks"

INPUT_GPKG_NAME = "new_blocks_populated.gpkg"
OUTPUT_GPKG_NAME = "new_blocks_populated.gpkg"

K_VALUES = [2, 3, 4, 5]

BLOCK_ID_FIELD = "block_id"
HETERO_FIELDS = ["HH_CC", "HH_Grtr10ha", "CC_Grtr10ha"]
LARGEPOP_FIELD = "LargePop"

POPULATION_FIELD = "population"
AREA_FIELD = "cell_area_m2"

POPULATION_THRESHOLD = 1000.0
AREA_THRESHOLD_M2 = 100000.0

OVERWRITE_OUTPUTS = True

LOG_PATH = OUTPUT_ROOT / "selection_log.txt"
SUMMARY_CSV = OUTPUT_ROOT / "selection_summary.csv"


# ------------------------------------------------------------
# Logging and helpers
# ------------------------------------------------------------

def log(message="", also_print=True):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    if also_print:
        print(message, flush=True)


def list_layer_names(dataset_path):
    layers = pyogrio.list_layers(str(dataset_path))
    if hasattr(layers, "shape"):
        return [str(row[0]) for row in layers]
    return [str(row[0]) if isinstance(row, (list, tuple)) else str(row) for row in layers]


def layer_exists(dataset_path, layer_name):
    target = layer_name.lower()
    return target in {layer.lower() for layer in list_layer_names(dataset_path)}


def actual_layer_name(dataset_path, layer_name):
    target = layer_name.lower()
    for layer in list_layer_names(dataset_path):
        if layer.lower() == target:
            return layer
    raise RuntimeError(f"Layer not found: {layer_name} in {dataset_path}")


def require_columns(df, required_columns, label):
    existing_lower = {c.lower() for c in df.columns}
    missing = [c for c in required_columns if c.lower() not in existing_lower]
    if missing:
        raise RuntimeError(f"{label} is missing required column(s): {missing}")


def actual_column_name(df, requested_name):
    requested_lower = requested_name.lower()
    for col in df.columns:
        if col.lower() == requested_lower:
            return col
    raise RuntimeError(f"Column not found: {requested_name}")


def flag_is_one(value):
    try:
        return int(float(value)) == 1
    except Exception:
        return False


def safe_float(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    try:
        return float(value)
    except Exception:
        return None


def block_folder_to_lookup_block_id(block_folder_name):
    return "blk_" + block_folder_name.lstrip("_")


def layer_base_name(layer_name):
    # Handles names displayed as main.blk_1_442_2.
    return layer_name.split(".")[-1]


def parse_k_from_layer(layer_name):
    base = layer_base_name(layer_name)
    last = base.split("_")[-1]
    return int(last) if last.isdigit() else None


def find_blk_layers_by_k(gpkg_path):
    out = {}
    for layer in list_layer_names(gpkg_path):
        base = layer_base_name(layer)
        if not base.lower().startswith("blk_"):
            continue
        k = parse_k_from_layer(base)
        if k in K_VALUES:
            out[k] = layer
    return out


# ------------------------------------------------------------
# Source membership lookup
# ------------------------------------------------------------

def build_hetero_orig(row):
    active = []
    for field in HETERO_FIELDS:
        if field in row.index and flag_is_one(row[field]):
            active.append(field)
    return "none" if not active else "|".join(active)


def read_source_membership_lookup():
    if not SOURCE_BLOCKS_GDB.exists():
        raise FileNotFoundError(f"Missing source GDB: {SOURCE_BLOCKS_GDB}")
    if not layer_exists(SOURCE_BLOCKS_GDB, SOURCE_BLOCKS_LAYER):
        raise FileNotFoundError(
            f"Missing source layer {SOURCE_BLOCKS_LAYER} in {SOURCE_BLOCKS_GDB}"
        )

    actual_layer = actual_layer_name(SOURCE_BLOCKS_GDB, SOURCE_BLOCKS_LAYER)
    columns = [BLOCK_ID_FIELD] + HETERO_FIELDS + [LARGEPOP_FIELD]

    log("Reading source block membership lookup...")
    log(f"  {SOURCE_BLOCKS_GDB} | {actual_layer}")

    source_df = pyogrio.read_dataframe(
        str(SOURCE_BLOCKS_GDB),
        layer=actual_layer,
        columns=columns,
        read_geometry=False,
    )

    require_columns(source_df, columns, SOURCE_BLOCKS_LAYER)

    lookup = {}
    duplicate_count = 0

    for _, row in source_df.iterrows():
        block_id = str(row[BLOCK_ID_FIELD]).strip()
        if not block_id:
            continue
        if block_id in lookup:
            duplicate_count += 1

        record = {field: 1 if flag_is_one(row[field]) else 0 for field in HETERO_FIELDS}
        record[LARGEPOP_FIELD] = 1 if flag_is_one(row[LARGEPOP_FIELD]) else 0
        record["hetero_orig"] = build_hetero_orig(row)
        record["large_pop_orig"] = record[LARGEPOP_FIELD]
        lookup[block_id] = record

    log(f"  Source records read: {len(source_df):,}")
    log(f"  Lookup records:      {len(lookup):,}")
    if duplicate_count:
        log(f"  WARNING: duplicate block_id values found: {duplicate_count:,}")
    return lookup


# ------------------------------------------------------------
# Layer evaluation and selection
# ------------------------------------------------------------

def evaluate_layer(gpkg_path, layer_name):
    gdf = gpd.read_file(str(gpkg_path), layer=layer_name)
    require_columns(gdf, [POPULATION_FIELD, AREA_FIELD], layer_name)

    pop_col = actual_column_name(gdf, POPULATION_FIELD)
    area_col = actual_column_name(gdf, AREA_FIELD)

    populations = gdf[pop_col].map(safe_float)
    areas = gdf[area_col].map(safe_float)

    nonnull_populations = populations.dropna()
    nonnull_areas = areas.dropna()
    n_features = len(gdf)

    if n_features == 0:
        return {
            "layer_name": layer_name,
            "n_features": 0,
            "max_population": None,
            "max_area_m2": None,
            "null_population_count": 0,
            "null_area_count": 0,
            "all_population_below_1000": False,
            "all_area_below_100000": False,
        }

    null_population_count = int(populations.isna().sum())
    null_area_count = int(areas.isna().sum())

    all_pop_below = (
        null_population_count == 0
        and bool((nonnull_populations < POPULATION_THRESHOLD).all())
    )
    all_area_below = (
        null_area_count == 0
        and bool((nonnull_areas < AREA_THRESHOLD_M2).all())
    )

    return {
        "layer_name": layer_name,
        "n_features": n_features,
        "max_population": float(nonnull_populations.max()) if len(nonnull_populations) else None,
        "max_area_m2": float(nonnull_areas.max()) if len(nonnull_areas) else None,
        "null_population_count": null_population_count,
        "null_area_count": null_area_count,
        "all_population_below_1000": all_pop_below,
        "all_area_below_100000": all_area_below,
    }


def choose_layer_for_block(gpkg_path, layer_by_k, membership):
    evaluations = {}
    for k in K_VALUES:
        if k not in layer_by_k:
            evaluations[k] = {"layer_name": None, "missing": True}
            continue
        stats = evaluate_layer(gpkg_path, layer_by_k[k])
        stats["missing"] = False
        evaluations[k] = stats

    large_pop = membership[LARGEPOP_FIELD] == 1
    hetero_any = any(membership[f] == 1 for f in HETERO_FIELDS)

    if large_pop:
        for k in K_VALUES:
            stats = evaluations.get(k, {})
            if not stats.get("missing") and stats["all_population_below_1000"]:
                return (
                    k,
                    stats["layer_name"],
                    f"LargePop=1; selected first k where all feature populations are < {POPULATION_THRESHOLD:g}.",
                    "",
                    evaluations,
                )

        stats5 = evaluations.get(5, {})
        if not stats5.get("missing") and stats5.get("all_area_below_100000"):
            return (
                5,
                stats5["layer_name"],
                f"LargePop=1; no k=2..5 layer had all populations < {POPULATION_THRESHOLD:g}; selected k=5 because all feature areas are < {AREA_THRESHOLD_M2:g} m2.",
                "",
                evaluations,
            )

        if not stats5.get("missing"):
            return (
                5,
                stats5["layer_name"],
                f"LargePop=1; no k=2..5 layer had all populations < {POPULATION_THRESHOLD:g}, and k=5 did not have all areas < {AREA_THRESHOLD_M2:g} m2; selected k=5 anyway.",
                "LargePop fallback warning: k=5 violates both preferred population and fallback area criteria.",
                evaluations,
            )

        available = [k for k in K_VALUES if not evaluations.get(k, {}).get("missing")]
        if available:
            k = max(available)
            stats = evaluations[k]
            return (
                k,
                stats["layer_name"],
                f"LargePop=1; k=5 layer missing, so selected highest available k={k} as emergency fallback.",
                "Missing k=5 fallback warning.",
                evaluations,
            )

        return (None, None, "LargePop=1 but no candidate blk_* layers were available.", "No layers available.", evaluations)

    if hetero_any:
        for k in K_VALUES:
            stats = evaluations.get(k, {})
            if not stats.get("missing") and stats["all_area_below_100000"]:
                return (
                    k,
                    stats["layer_name"],
                    f"LargePop!=1 and heterogeneous; selected first k where all feature areas are < {AREA_THRESHOLD_M2:g} m2.",
                    "",
                    evaluations,
                )

        stats5 = evaluations.get(5, {})
        if not stats5.get("missing"):
            return (
                5,
                stats5["layer_name"],
                f"LargePop!=1 and heterogeneous; no k=2..5 layer had all feature areas < {AREA_THRESHOLD_M2:g} m2; selected k=5 anyway.",
                "Heterogeneous fallback warning: k=5 violates area criterion.",
                evaluations,
            )

        available = [k for k in K_VALUES if not evaluations.get(k, {}).get("missing")]
        if available:
            k = max(available)
            stats = evaluations[k]
            return (
                k,
                stats["layer_name"],
                f"LargePop!=1 and heterogeneous; k=5 layer missing, so selected highest available k={k} as emergency fallback.",
                "Missing k=5 fallback warning.",
                evaluations,
            )

        return (None, None, "Heterogeneous non-LargePop block but no candidate blk_* layers were available.", "No layers available.", evaluations)

    return (
        None,
        None,
        "Block is neither LargePop nor heterogeneous according to lookup; skipped.",
        "Skipped: no relevant origin membership.",
        evaluations,
    )


# ------------------------------------------------------------
# Output helpers
# ------------------------------------------------------------

def copy_selected_layer(input_gpkg, input_layer, output_block_folder, output_layer):
    output_block_folder.mkdir(parents=True, exist_ok=True)
    output_gpkg = output_block_folder / OUTPUT_GPKG_NAME

    if OVERWRITE_OUTPUTS and output_gpkg.exists():
        output_gpkg.unlink()

    gdf = gpd.read_file(str(input_gpkg), layer=input_layer)
    gdf.to_file(str(output_gpkg), layer=output_layer, driver="GPKG", engine="pyogrio")
    return output_gpkg


def evaluation_value(evaluations, k, key):
    stats = evaluations.get(k, {})
    if stats.get("missing"):
        return None
    return stats.get(key)


def add_evaluation_fields(row, evaluations):
    for k in K_VALUES:
        row[f"k{k}_n_features"] = evaluation_value(evaluations, k, "n_features")
        row[f"k{k}_max_population"] = evaluation_value(evaluations, k, "max_population")
        row[f"k{k}_max_area_m2"] = evaluation_value(evaluations, k, "max_area_m2")
        row[f"k{k}_all_population_below_1000"] = evaluation_value(evaluations, k, "all_population_below_1000")
        row[f"k{k}_all_area_below_100000"] = evaluation_value(evaluations, k, "all_area_below_100000")
    return row


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    if LOG_PATH.exists():
        LOG_PATH.unlink()

    log("Starting new-block selection workflow")
    log(f"Input root:  {INPUT_ROOT}")
    log(f"Output root: {OUTPUT_ROOT}")
    log("")

    if not INPUT_ROOT.exists():
        raise FileNotFoundError(f"Input root does not exist: {INPUT_ROOT}")

    source_lookup = read_source_membership_lookup()
    summary_rows = []

    block_folders = [p for p in sorted(INPUT_ROOT.iterdir()) if p.is_dir() and p.name.startswith("_")]

    log("")
    log(f"Block folders found: {len(block_folders):,}")
    log("")

    for block_folder in block_folders:
        block_folder_name = block_folder.name
        lookup_block_id = block_folder_to_lookup_block_id(block_folder_name)
        input_gpkg = block_folder / INPUT_GPKG_NAME
        output_block_folder = OUTPUT_ROOT / block_folder_name

        log("=" * 80)
        log(f"Block folder: {block_folder_name}")
        log(f"Lookup block_id: {lookup_block_id}")

        row_base = {
            "block_folder": block_folder_name,
            "lookup_block_id": lookup_block_id,
            "input_gpkg": str(input_gpkg),
            "output_folder": str(output_block_folder),
        }

        try:
            if not input_gpkg.exists():
                reason = f"Missing input GeoPackage: {input_gpkg}"
                log(f"  SKIP: {reason}")
                summary_rows.append({**row_base, "status": "skipped", "selected_k": None, "selected_layer": None, "selection_reason": reason, "warning": "missing input gpkg"})
                continue

            if lookup_block_id not in source_lookup:
                reason = f"block_id {lookup_block_id} not found in source lookup"
                log(f"  SKIP: {reason}")
                summary_rows.append({**row_base, "status": "skipped", "selected_k": None, "selected_layer": None, "selection_reason": reason, "warning": "missing source lookup record"})
                continue

            membership = source_lookup[lookup_block_id]
            log(
                "  Membership: "
                f"HH_CC={membership['HH_CC']}, "
                f"HH_Grtr10ha={membership['HH_Grtr10ha']}, "
                f"CC_Grtr10ha={membership['CC_Grtr10ha']}, "
                f"LargePop={membership['LargePop']}, "
                f"hetero_orig={membership['hetero_orig']}"
            )

            layer_by_k = find_blk_layers_by_k(input_gpkg)
            log(f"  Candidate layers found by k: {sorted(layer_by_k.keys())}")

            selected_k, selected_layer, reason, warning, evaluations = choose_layer_for_block(
                gpkg_path=input_gpkg,
                layer_by_k=layer_by_k,
                membership=membership,
            )

            if selected_layer is None:
                log(f"  SKIP: {reason}")
                if warning:
                    log(f"  WARNING: {warning}")
                row = {
                    **row_base,
                    "status": "skipped",
                    "selected_k": None,
                    "selected_layer": None,
                    "selection_reason": reason,
                    "warning": warning,
                    "HH_CC": membership["HH_CC"],
                    "HH_Grtr10ha": membership["HH_Grtr10ha"],
                    "CC_Grtr10ha": membership["CC_Grtr10ha"],
                    "LargePop": membership["LargePop"],
                    "hetero_orig": membership["hetero_orig"],
                    "large_pop_orig": membership["large_pop_orig"],
                }
                summary_rows.append(add_evaluation_fields(row, evaluations))
                continue

            log(f"  Selected k={selected_k}: {selected_layer}")
            log(f"  Reason: {reason}")
            if warning:
                log(f"  WARNING: {warning}")

            output_gpkg = copy_selected_layer(
                input_gpkg=input_gpkg,
                input_layer=selected_layer,
                output_block_folder=output_block_folder,
                output_layer=layer_base_name(selected_layer),
            )
            log(f"  Wrote selected layer to: {output_gpkg}")

            row = {
                **row_base,
                "status": "selected",
                "selected_k": selected_k,
                "selected_layer": selected_layer,
                "output_gpkg": str(output_gpkg),
                "selection_reason": reason,
                "warning": warning,
                "HH_CC": membership["HH_CC"],
                "HH_Grtr10ha": membership["HH_Grtr10ha"],
                "CC_Grtr10ha": membership["CC_Grtr10ha"],
                "LargePop": membership["LargePop"],
                "hetero_orig": membership["hetero_orig"],
                "large_pop_orig": membership["large_pop_orig"],
            }
            summary_rows.append(add_evaluation_fields(row, evaluations))

        except Exception:
            error_text = traceback.format_exc()
            log("  ERROR:")
            log(error_text)
            summary_rows.append({**row_base, "status": "error", "selected_k": None, "selected_layer": None, "selection_reason": "error while processing block", "warning": error_text})

    log("")
    log("Writing selection summary CSV...")
    log(f"  {SUMMARY_CSV}")

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8")

    selected_count = int((summary_df["status"] == "selected").sum()) if "status" in summary_df.columns else 0
    skipped_count = int((summary_df["status"] == "skipped").sum()) if "status" in summary_df.columns else 0
    error_count = int((summary_df["status"] == "error").sum()) if "status" in summary_df.columns else 0

    log("")
    log("Finished.")
    log(f"Blocks processed: {len(summary_rows):,}")
    log(f"Selected blocks:  {selected_count:,}")
    log(f"Skipped blocks:   {skipped_count:,}")
    log(f"Errored blocks:   {error_count:,}")


if __name__ == "__main__":
    main()
